# Stage 1 -- Java Dataset Statistics

Produces `outputs/stage1_java_stats.xlsx` with sheets: **Java_Real**, **Java_Synth**, **Java_AI**, **Filters**, **Legend**.

**Prerequisites:** run `00_download_datasets.ipynb` first.

In [1]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW      = ROOT / 'data' / 'raw'
CWE_XML  = ROOT / 'data' / 'cwec_latest.xml'
OUT_DIR  = ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
XLSX_PATH = OUT_DIR / 'stage1_java_stats.xlsx'

DATASET_FILTERS = {
    'CVEfixes(Java)':  {'branch': 'real',  'language': 'Java', 'source': 'HuggingFace hitoshura25/cvefixes', 'positives': 'language==Java, vulnerable_code non-empty, CWE non-empty, single-function commit', 'negatives': 'none', 'cwe_source': 'cwe_id column (NVD join)', 'filters_applied': 'language filter, NVD placeholder drop, single-function commit', 'safe_type': 'none', 'label_quality': '3/5', 'notes': ''},
    'CrossVul(Java)':  {'branch': 'real',  'language': 'Java', 'source': 'Zenodo crossvul.zip', 'positives': 'bad_* files (vulnerable functions)', 'negatives': 'good_* files (fixed functions)', 'cwe_source': 'CWE-NNN directory name', 'filters_applied': 'java/ folder filter', 'safe_type': 'fix-paired', 'label_quality': '3/5', 'notes': ''},
    'Juliet(Java)':    {'branch': 'synth', 'language': 'Java', 'source': 'NIST SARD juliet-test-suite-for-java.zip', 'positives': '*_bad.java files', 'negatives': '*_good*.java files', 'cwe_source': 'directory name (CWE-NNN prefix)', 'filters_applied': 'filename pattern', 'safe_type': 'pure', 'label_quality': '5/5', 'notes': 'whole .java file treated as sample'},
    'OWASP(Java)':     {'branch': 'synth', 'language': 'Java', 'source': 'OWASP GitHub BenchmarkJava', 'positives': 'real vulnerability=true in expectedresults-1.2.csv', 'negatives': 'real vulnerability=false', 'cwe_source': 'cwe column (bare digit, normalised to CWE-NNN)', 'filters_applied': 'CSV join on filename (case-insensitive stem match)', 'safe_type': 'pure', 'label_quality': '5/5', 'notes': ''},
    'CAPEC_LLM(Java)': {'branch': 'ai',   'language': 'Java', 'source': 'GitHub llmForCapec/CAPECDatasetsLLM', 'positives': 'code_snippet detected as Java, CWE from description', 'negatives': 'none', 'cwe_source': 'CWE-NNN regex on description field', 'filters_applied': 'language detection (Java keywords), CWE regex match', 'safe_type': 'none', 'label_quality': '2/5', 'notes': 'LLM-generated; CWE from CAPEC description, not per-sample'},
}

print(f'Root: {ROOT}')
print(f'Output: {XLSX_PATH}')

Root: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST
Output: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\outputs\stage1_java_stats.xlsx


## 1. CWE XML -- download if missing

In [2]:
import io, urllib.request, zipfile

CWE_ZIP_URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'

if CWE_XML.exists():
    print(f'CWE XML present ({CWE_XML.stat().st_size / 1e6:.1f} MB) -- skipping download.')
else:
    print('Downloading CWE XML from MITRE ...')
    with urllib.request.urlopen(CWE_ZIP_URL, timeout=60) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        xml_name = next(n for n in zf.namelist() if n.endswith('.xml'))
        CWE_XML.write_bytes(zf.read(xml_name))
    print(f'Saved: {CWE_XML} ({CWE_XML.stat().st_size / 1e6:.1f} MB)')

CWE XML present (16.1 MB) -- skipping download.


## 2. Load CWE Navigator

In [3]:
from ingestion.cwe_navigator import CWENavigator

nav = CWENavigator(str(CWE_XML))
print(f'Loaded: {len(nav.weaknesses):,} weaknesses, {len(nav.categories):,} categories')

_parent_ids: set[str] = {p for parents in nav.child_of.values() for p in parents}

def _strip(cwe_str: str) -> str:
    return cwe_str.removeprefix('CWE-').removeprefix('cwe-').strip()

def cwe_label(cwe_str: str) -> str:
    num = _strip(cwe_str)
    if num in nav.categories or num in nav.views:
        return 'category'
    if num not in nav.weaknesses:
        return 'unknown'
    if nav.get_element_name(num).startswith('DEPRECATED:'):
        return 'deprecated'
    return 'leaf' if num not in _parent_ids else 'non-leaf'

Loaded: 969 weaknesses, 420 categories


## 3. Extract samples (skip if files missing)

In [4]:
from collections import Counter
from ingestion.schema          import FunctionSample
from ingestion.cvefixes        import extract_cvefixes
from ingestion.crossvul        import extract_crossvul
from ingestion.juliet          import extract_juliet
from ingestion.owasp_benchmark import extract_owasp_benchmark
from ingestion.capec_llm       import extract_capec_llm

def _try(name, loader):
    try:
        s = loader()
        print(f'  {name:22s}: {len(s):>7,} samples')
        return s
    except FileNotFoundError as e:
        print(f'  {name:22s}: SKIPPED -- {e}')
        return []

collections: dict[str, list[FunctionSample]] = {}

collections['CVEfixes(Java)']  = _try('CVEfixes(Java)',  lambda: extract_cvefixes(RAW/'cvefixes', language='Java'))
collections['CrossVul(Java)']  = _try('CrossVul(Java)',  lambda: extract_crossvul(RAW/'crossvul.zip', language='Java'))
collections['Juliet(Java)']    = _try('Juliet(Java)',    lambda: extract_juliet(RAW/'juliet_java.zip', language='Java'))
collections['OWASP(Java)']     = _try('OWASP(Java)',     lambda: extract_owasp_benchmark(RAW/'owasp_benchmark', language='Java'))
collections['CAPEC_LLM(Java)'] = _try('CAPEC_LLM(Java)', lambda: extract_capec_llm(RAW/'capec_llm', language='Java'))

collections = {k: v for k, v in collections.items() if v}
print(f'\nLoaded: {list(collections.keys())}')

  CVEfixes(Java)        :     324 samples
  CrossVul(Java)        :   1,126 samples
  Juliet(Java)          :   2,035 samples
  OWASP(Java)           :   2,740 samples
  CAPEC_LLM(Java)       :   3,625 samples

Loaded: ['CVEfixes(Java)', 'CrossVul(Java)', 'Juliet(Java)', 'OWASP(Java)', 'CAPEC_LLM(Java)']


## 4. Compute statistics

In [5]:
stats: dict[str, dict] = {}

for name, samples in collections.items():
    vuln    = [s for s in samples if s.label == 1]
    safe    = [s for s in samples if s.label == 0]
    counter = Counter(cwe for s in vuln for cwe in s.cwes)
    branch  = samples[0].branch if samples else ''
    stats[name] = {
        'total': len(samples), 'vulnerable': len(vuln), 'safe': len(safe),
        'unique_cwes': len(counter), 'cwe_counts': counter, 'branch': branch,
    }

print(f'{"Dataset":<24} {"Branch":>6} {"Total":>8} {"Vuln":>8} {"Safe":>8} {"CWEs":>6}')
print('-' * 66)
for n, s in stats.items():
    print(f'{n:<24} {s["branch"]:>6} {s["total"]:>8,} {s["vulnerable"]:>8,} {s["safe"]:>8,} {s["unique_cwes"]:>6,}')

Dataset                  Branch    Total     Vuln     Safe   CWEs
------------------------------------------------------------------
CVEfixes(Java)             real      324      324        0     72
CrossVul(Java)             real    1,126      563      563     47
Juliet(Java)              synth    2,035      750    1,285     40
OWASP(Java)               synth    2,740    1,415    1,325     11
CAPEC_LLM(Java)              ai    3,625    3,625        0    516


## 5. Build Excel report

In [6]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FILL_HEADER     = PatternFill('solid', fgColor='1F4E79')
FILL_LEAF       = PatternFill('solid', fgColor='92D050')
FILL_NONLEAF    = PatternFill('solid', fgColor='FFC000')
FILL_UNKNOWN    = PatternFill('solid', fgColor='BFBFBF')
FILL_DEPRECATED = PatternFill('solid', fgColor='FFB3B3')
FILL_CATEGORY   = PatternFill('solid', fgColor='D2B4DE')
FILL_ROW_ODD    = PatternFill('solid', fgColor='F2F2F2')
FONT_HEADER     = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
FONT_CWE        = Font(bold=True, color='000000', name='Calibri', size=10)
FONT_DATA       = Font(name='Calibri', size=10)
FONT_DATA_B     = Font(bold=True, name='Calibri', size=10)
ALIGN_C = Alignment(horizontal='center', vertical='center', wrap_text=True)
ALIGN_L = Alignment(horizontal='left', vertical='center')
THIN    = Side(style='thin', color='D3D3D3')
BORDER  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

_CWE_FILL = {'leaf': FILL_LEAF, 'non-leaf': FILL_NONLEAF, 'unknown': FILL_UNKNOWN,
             'deprecated': FILL_DEPRECATED, 'category': FILL_CATEGORY}

FIXED_COLS = ['Dataset', 'Total Samples', 'Vulnerable', 'Safe', 'Unique CWEs']
N_FIXED    = len(FIXED_COLS)

def _write_data_sheet(ws, dataset_names, all_stats):
    global_c = Counter()
    for dn in dataset_names:
        global_c.update(all_stats[dn]['cwe_counts'])
    cwes = [cwe for cwe, _ in global_c.most_common()]

    for ci, col in enumerate(FIXED_COLS, 1):
        cell = ws.cell(1, ci, col)
        cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER

    for ci, cwe in enumerate(cwes, N_FIXED + 1):
        num  = _strip(cwe)
        name = nav.get_element_name(num)
        cell = ws.cell(1, ci, f'{cwe}\n{name}' if name != 'Unknown' else cwe)
        cell.fill, cell.font, cell.alignment, cell.border = _CWE_FILL[cwe_label(cwe)], FONT_CWE, ALIGN_C, BORDER

    for ri, dn in enumerate(dataset_names, 2):
        s = all_stats[dn]
        rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
        for ci, val in enumerate([dn, s['total'], s['vulnerable'], s['safe'], s['unique_cwes']], 1):
            cell = ws.cell(ri, ci, val)
            cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
            cell.alignment = ALIGN_L if ci == 1 else ALIGN_C
            cell.fill, cell.border = rf, BORDER
        for ci, cwe in enumerate(cwes, N_FIXED + 1):
            cnt = s['cwe_counts'].get(cwe, 0)
            cell = ws.cell(ri, ci, cnt if cnt > 0 else None)
            cell.font, cell.alignment, cell.fill, cell.border = FONT_DATA, ALIGN_C, rf, BORDER

    ws.column_dimensions['A'].width = 22
    for c in range(2, N_FIXED + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    for c in range(N_FIXED + 1, N_FIXED + len(cwes) + 1):
        ws.column_dimensions[get_column_letter(c)].width = 16
    ws.row_dimensions[1].height = 48
    ws.freeze_panes = 'B2'

wb = Workbook()
wb.remove(wb.active)

SHEET_MAP = {'real': 'Java_Real', 'synth': 'Java_Synth', 'ai': 'Java_AI'}
for branch, sheet_name in SHEET_MAP.items():
    datasets = [n for n, s in stats.items() if s['branch'] == branch]
    if datasets:
        ws = wb.create_sheet(sheet_name)
        _write_data_sheet(ws, datasets, stats)

wf = wb.create_sheet('Filters')
filter_cols = ['Dataset', 'Branch', 'Language', 'Source', 'Positives', 'Negatives',
               'CWE Source', 'Filters Applied', 'Safe Type', 'Label Quality', 'Notes']
for ci, col in enumerate(filter_cols, 1):
    cell = wf.cell(1, ci, col)
    cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER
for ri, (ds_name, filt) in enumerate(DATASET_FILTERS.items(), 2):
    rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
    row_vals = [ds_name, filt['branch'], filt['language'], filt['source'],
                filt['positives'], filt['negatives'], filt['cwe_source'],
                filt['filters_applied'], filt['safe_type'], filt['label_quality'], filt['notes']]
    for ci, val in enumerate(row_vals, 1):
        cell = wf.cell(ri, ci, val)
        cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
        cell.alignment = ALIGN_L
        cell.fill, cell.border = rf, BORDER
filter_widths = [22, 8, 8, 35, 45, 30, 30, 45, 14, 12, 40]
for ci, w in enumerate(filter_widths, 1):
    wf.column_dimensions[get_column_letter(ci)].width = w
wf.freeze_panes = 'B2'

wl = wb.create_sheet('Legend')
legend_rows = [
    ('Colour', 'Meaning'),
    ('Green  (#92D050)', 'Leaf CWE -- most specific; no children in MITRE tree'),
    ('Amber  (#FFC000)', 'Non-leaf CWE -- parent/intermediate node'),
    ('Gray   (#BFBFBF)', 'Unknown CWE -- ID not in cwec_latest.xml'),
    ('Pink   (#FFB3B3)', 'Deprecated CWE -- name starts with DEPRECATED:'),
    ('Purple (#D2B4DE)', 'Category/View -- MITRE organisational grouping'),
]
fills = [FILL_HEADER, FILL_LEAF, FILL_NONLEAF, FILL_UNKNOWN, FILL_DEPRECATED, FILL_CATEGORY]
for ri, (a, b) in enumerate(legend_rows, 1):
    ca, cb = wl.cell(ri, 1, a), wl.cell(ri, 2, b)
    ca.fill = fills[ri - 1]
    ca.font = FONT_HEADER if ri == 1 else FONT_CWE
    cb.font = FONT_HEADER if ri == 1 else FONT_DATA
    ca.alignment = cb.alignment = ALIGN_L
    ca.border = cb.border = BORDER
wl.column_dimensions['A'].width = 22
wl.column_dimensions['B'].width = 60

wb.save(XLSX_PATH)
print(f'Saved: {XLSX_PATH}')
print(f'Sheets: {wb.sheetnames}')

Saved: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\outputs\stage1_java_stats.xlsx
Sheets: ['Java_Real', 'Java_Synth', 'Java_AI', 'Filters', 'Legend']


## 6. Summary

In [7]:
global_counter: Counter = Counter()
for s in stats.values():
    global_counter.update(s['cwe_counts'])
ALL_CWES = [cwe for cwe, _ in global_counter.most_common()]

by_label = {lbl: [c for c in ALL_CWES if cwe_label(c) == lbl]
            for lbl in ('leaf', 'non-leaf', 'unknown', 'deprecated', 'category')}
for lbl, cwes in by_label.items():
    print(f'  {lbl:<12}: {len(cwes):>4}  {cwes[:5]} ...')

  leaf        :  356  ['CWE-78', 'CWE-494', 'CWE-924', 'CWE-294', 'CWE-941'] ...
  non-leaf    :  176  ['CWE-79', 'CWE-200', 'CWE-89', 'CWE-506', 'CWE-319'] ...
  unknown     :    0  [] ...
  deprecated  :   12  ['CWE-592', 'CWE-217', 'CWE-247', 'CWE-534', 'CWE-218'] ...
  category    :    7  ['CWE-264', 'CWE-310', 'CWE-361', 'CWE-19', 'CWE-320'] ...
